# Phase 6 - Geographic + Census Join

Goal: assign each complaint to a NYC community district via spatial join, then compute per-district fingerprints — complaint volume, top categories, and the most distinctive terms (TF-IDF lift over corpus). These artifacts power the City Pulse and Cluster Atlas dashboard tabs.

We use geopandas on the driver instead of Spark for the spatial join. There are only 71 community districts; pulling the lat/long of 1.94M complaints to pandas is faster than configuring Sedona.

Phase 2's `sample_2m_preprocessed.parquet` must be on Drive.

## Cell 1 - Bootstrap

In [ ]:
REPO_URL = 'https://github.com/george-gideon-S/cs-gy-6513-big-data-311-nlp.git'

from google.colab import drive
drive.mount('/content/drive')

import subprocess, os, sys
if not os.path.isdir('/content/project/.git'):
    subprocess.run(['git', 'clone', REPO_URL, '/content/project'], check=True)
else:
    subprocess.run(['git', '-C', '/content/project', 'pull'], check=True)

if '/content/project' not in sys.path:
    sys.path.insert(0, '/content/project')

!pip install -r /content/project/requirements.txt -q

!apt-get install -y openjdk-11-jre-headless > /dev/null 2>&1
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-11-openjdk-amd64'
os.environ['PATH'] = os.environ['JAVA_HOME'] + '/bin:' + os.environ['PATH']

import nltk
for pkg in ['stopwords', 'wordnet', 'punkt', 'punkt_tab', 'omw-1.4']:
    nltk.download(pkg, download_dir='/root/nltk_data', quiet=True)

from src.spark_setup import get_spark
spark = get_spark(app_name='phase6-geo')
print('spark', spark.version, 'ready')

## Cell 2 - Download community districts GeoJSON

NYC Open Data ID `mzpm-a6vd` — 71 polygons covering all 5 boroughs.

In [ ]:
import requests
from pathlib import Path

geo_path = Path('/content/project/dashboard/assets/community_districts.geojson')
if not geo_path.exists():
    url = 'https://data.cityofnewyork.us/resource/jp9i-3b7y.geojson'
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    geo_path.parent.mkdir(parents=True, exist_ok=True)
    geo_path.write_bytes(r.content)
    print(f'downloaded {len(r.content):,} bytes -> {geo_path}')
else:
    print(f'already on disk: {geo_path} ({geo_path.stat().st_size:,} bytes)')

## Cell 3 - Inspect the polygons

In [ ]:
import geopandas as gpd

gdf = gpd.read_file(geo_path)
print(f'loaded {len(gdf)} polygons')
print('columns:', list(gdf.columns))
print('crs:', gdf.crs)
print('sample rows:')
print(gdf[['boro_cd']].head() if 'boro_cd' in gdf.columns else gdf.head())

# convert to lat/long if needed
if gdf.crs is None or gdf.crs.to_string() != 'EPSG:4326':
    gdf = gdf.to_crs('EPSG:4326')
    print('reprojected to EPSG:4326')

## Cell 4 - Pull lat/long + minimal metadata to driver pandas

We need lat/long, label_canonical, and tokens to do the per-district aggregations. Skip rows without coords.

In [ ]:
from pyspark.sql import functions as F

in_path = '/content/drive/MyDrive/cs6513/sample_2m_preprocessed.parquet'
df = (
    spark.read.parquet(in_path)
    .filter(F.size('tokens') > 0)
    .filter(F.col('latitude').isNotNull() & F.col('longitude').isNotNull())
    .filter(F.col('latitude').between(40.4, 41.0))   # rough nyc bounds
    .filter(F.col('longitude').between(-74.5, -73.5))
    .select('unique_key', 'label_canonical', 'borough', 'tokens', 'latitude', 'longitude')
)
n_geo = df.count()
print(f'rows with valid nyc lat/long: {n_geo:,}')

## Cell 5 - Spatial join: assign each complaint to a community district

We pull to pandas + geopandas because spatial joins on Spark require Sedona which is heavyweight setup. With ~1.9M points this takes about a minute on driver.

In [ ]:
from shapely.geometry import Point
import time

# pull only the columns we need for join (keep memory under 1GB)
t0 = time.time()
pdf = df.select('unique_key', 'latitude', 'longitude').toPandas()
print(f'pulled {len(pdf):,} rows in {time.time()-t0:.1f} sec')

# build geometries
t0 = time.time()
pdf['geometry'] = [Point(xy) for xy in zip(pdf['longitude'], pdf['latitude'])]
points_gdf = gpd.GeoDataFrame(pdf, geometry='geometry', crs='EPSG:4326')
print(f'built geometries in {time.time()-t0:.1f} sec')

# spatial join
t0 = time.time()
joined = gpd.sjoin(
    points_gdf, gdf[['boro_cd', 'geometry']],
    how='left', predicate='within',
)
joined = joined[['unique_key', 'boro_cd']].rename(columns={'boro_cd': 'district'})
joined['district'] = joined['district'].astype('Int64')  # keep nullable int
print(f'sjoin in {time.time()-t0:.1f} sec')

n_unmatched = joined['district'].isna().sum()
print(f'matched {(joined["district"].notna()).sum():,} rows; unmatched (in water etc.): {n_unmatched:,}')

## Cell 6 - Push the assignments back to Spark for aggregation

We re-attach the district column to the original Spark df so subsequent aggregations stay in Spark.

In [ ]:
joined_clean = joined.dropna(subset=['district']).copy()
joined_clean['district'] = joined_clean['district'].astype(int)

joined_sdf = spark.createDataFrame(joined_clean[['unique_key', 'district']])
df_geo = df.join(joined_sdf, on='unique_key', how='inner').cache()
n_geo_final = df_geo.count()
print(f'rows with district assignments: {n_geo_final:,}')

## Cell 7 - Per-district complaint volume + top categories

This drives the City Pulse choropleth tile.

In [ ]:
import pandas as pd

# volume per district
volume = (
    df_geo.groupBy('district').count()
    .withColumnRenamed('count', 'complaint_count')
    .toPandas()
    .sort_values('complaint_count', ascending=False)
)
print('top 10 districts by complaint volume:')
print(volume.head(10).to_string(index=False))

# top 3 categories per district
cat_counts = (
    df_geo.groupBy('district', 'label_canonical').count()
    .toPandas()
)
top_cats_per_district = {}
for district, sub in cat_counts.groupby('district'):
    sub = sub.sort_values('count', ascending=False).head(3)
    top_cats_per_district[int(district)] = [
        (row['label_canonical'], int(row['count'])) for _, row in sub.iterrows()
    ]

print(f'\nsample - top 3 categories for first 5 districts:')
for d in list(top_cats_per_district.keys())[:5]:
    cats = ', '.join(f'{n} ({c})' for n, c in top_cats_per_district[d])
    print(f'  district {d}: {cats}')

## Cell 8 - Per-district language fingerprint via TF-IDF lift

Lift = (term frequency in district) / (term frequency in whole corpus). High lift means a term is *distinctively used* in that district vs the city average. This surfaces neighborhood character — a district with high lift on `rat` and `garbage` reads as a different urban issue profile than one with high lift on `tree` and `branch`.

In [ ]:
# explode tokens with district
exploded = df_geo.select('district', F.explode('tokens').alias('term'))

# tf per (district, term)
by_district = (
    exploded.groupBy('district', 'term').count()
    .withColumnRenamed('count', 'tf_district')
)

# tf corpus-wide
by_corpus = (
    exploded.groupBy('term').count()
    .withColumnRenamed('count', 'tf_corpus')
)
corpus_total = by_corpus.agg(F.sum('tf_corpus')).collect()[0][0]
by_corpus = by_corpus.withColumn('tf_corpus_norm', F.col('tf_corpus') / F.lit(corpus_total))

# district totals for normalization
district_totals = (
    by_district.groupBy('district').agg(F.sum('tf_district').alias('d_total'))
)

# join everything and compute lift
joined_tf = (
    by_district.join(by_corpus, on='term', how='left')
    .join(district_totals, on='district', how='left')
    .withColumn('tf_district_norm', F.col('tf_district') / F.col('d_total'))
    .withColumn('lift', F.col('tf_district_norm') / F.col('tf_corpus_norm'))
)

# require minimum tf so we dont surface typos that lift to infinity on tiny counts
joined_tf = joined_tf.filter(F.col('tf_district') >= 50)

lift_pdf = joined_tf.select('district', 'term', 'tf_district', 'lift').toPandas()
print(f'computed lift for {len(lift_pdf):,} (district, term) pairs')

## Cell 9 - Top 10 distinctive terms per district

Surface the 10 highest-lift terms per district. These become the per-district word clouds in the Cluster Atlas tab.

In [ ]:
fingerprints = {}
for district, sub in lift_pdf.groupby('district'):
    top = sub.nlargest(10, 'lift')
    fingerprints[int(district)] = [
        (row['term'], round(float(row['lift']), 2), int(row['tf_district']))
        for _, row in top.iterrows()
    ]

print(f'fingerprinted {len(fingerprints)} districts')

# show 5 districts as a sanity check
import random
sample_districts = random.sample(sorted(fingerprints.keys()), min(5, len(fingerprints)))
print('\nsample fingerprints:')
for d in sample_districts:
    terms = ', '.join(f'{t}({l})' for t, l, _ in fingerprints[d][:6])
    print(f'  district {d}: {terms}')

## Cell 10 - Save artifacts for the dashboard

Three small JSON files plus a parquet of district volume + lift table. Streamlit Cloud reads from the JSONs on app start.

In [ ]:
import json, datetime

# 1. district volume + top categories
volume_dict = {
    str(int(row['district'])): {
        'count': int(row['complaint_count']),
        'top_categories': top_cats_per_district[int(row['district'])],
    }
    for _, row in volume.iterrows()
}
with open('/content/project/dashboard/assets/district_volume.json', 'w') as f:
    json.dump(volume_dict, f, indent=2, default=str)
print('saved district_volume.json')

# 2. per-district fingerprints
fingerprints_jsonable = {str(k): v for k, v in fingerprints.items()}
with open('/content/project/dashboard/assets/district_fingerprints.json', 'w') as f:
    json.dump(fingerprints_jsonable, f, indent=2, default=str)
print('saved district_fingerprints.json')

# 3. summary metadata
summary = {
    'phase': 6,
    'computed_at': datetime.datetime.utcnow().isoformat() + 'Z',
    'rows_geocoded': int(n_geo_final),
    'rows_unmatched_to_district': int(n_unmatched),
    'n_districts_with_data': len(fingerprints),
    'top_5_districts_by_volume': [
        {'district': int(row['district']), 'count': int(row['complaint_count'])}
        for _, row in volume.head(5).iterrows()
    ],
}
with open('/content/project/dashboard/assets/geo_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print('saved geo_summary.json')

# 4. district volume parquet on drive (for any large-data analysis later)
volume.to_parquet('/content/drive/MyDrive/cs6513/district_volume.parquet')
print('saved district_volume.parquet to drive')

## Cell 11 - Push artifacts to GitHub

Commits the four geo artifacts (geojson + 3 JSONs) to the repo so Streamlit Cloud can read them at deploy time. Requires `GITHUB_PAT` in Colab Secrets (key icon in sidebar).

In [ ]:
from src.colab_git import commit_artifacts
commit_artifacts(message='phase 6: geographic + census join artifacts')

## Phase 6 - Done when

- Cell 5 reports >95% of nyc lat/long rows assigned to a district (unmatched are in water bodies or just-outside-nyc edge cases).
- Cell 7 prints top-10 districts by volume (Manhattan and Brooklyn districts dominate as expected).
- Cell 9 prints distinctive-term fingerprints that pass a sanity check (e.g., one district leads with `rat`, another with `tree branch`).
- `dashboard/assets/{community_districts.geojson, district_volume.json, district_fingerprints.json, geo_summary.json}` all exist.

Save the print as `PRINT 7.pdf` and drop in the project directory. Phase 6 unlocks the City Pulse dashboard tab.